# iCESM1.2 LIG (127 ka) $-$ PI

Companion to `LGM_analyses.ipynb`, built the same way: per-year monthly output for both cases,
precipitation-weighted isotope fields, and LIG$-$PI differences masked for statistical
significance.

## The two cases

| | LIG | PI |
|---|---|---|
| case | `b.e12.B1850C5.f19_g16.iLIG127k.001` | `b.e12.B1850C5.f19_g16.iPI.01` |
| years | 401$-$500 (the run's last century; it ends at 501) | 801$-$900 |
| source | paleo-calendar-adjusted files, see below | `data/raw/*.iPI.01.0801-0900.tseries.nc` |

The LIG run was branched from `iPI.01` at year 901 — its history files name
`b.e12.B1850C5.f19_g16.iPI.01.cam.i.0901-01-01-00000.nc` as `initial_file` — so PI years
801$-$900 are the parent control's last century before the branch. They are the same files
`LGM_analyses.ipynb` reads, so both model notebooks share one PI baseline.

## The paleo calendar adjustment

A paleoclimate run's months do not line up with modern ones. Under 127 ka orbital forcing,
precession shifts the equinoxes and solstices relative to the calendar, so a "July" mean from
the raw model output covers a different slice of the seasonal cycle than a modern July does.
Comparing it against a PI July, or against a proxy calibrated on the modern seasonal cycle,
compares two different intervals. **PaleoCalAdjust** corrects for this: it interpolates the
monthly means to pseudo-daily values, re-integrates them over the paleo month boundaries, and
writes monthly values that are directly comparable to a fixed modern calendar.

| | |
|---|---|
| Program | PaleoCalAdjust v1.1, `cal_adjust.f90` (Bartlein & Shafer 2019, *Geosci. Model Dev.* 12, 3889–3913) |
| Local copy | `$WORK_DATA_DIR/PaleoCalAdjust/` |
| Info file | `data/info_files/cal_adj_info_lig127ka.csv` |
| Input | `$WORK_DATA_DIR/LIG/tseries_last_100_yrs/{atm.2d.vars,dh.precIsotopes,o.precIsotopes}.iLIG127k.tseries.nc` |
| Output | `{varn}_Amon_CESM1.2_LIG127k_146031-182500_cal_adj.nc`, 1200 monthly records |
| Settings | age 127 ka, `noleap`, Harzallah pseudo-daily interpolation, `match_mean=TRUE`, `tol=0.01`, no negative-value clipping |

**Nineteen variables were adjusted:** the 16 precipitation isotope tracers, `PRECC`, `PRECL`,
and `TS`; `U`, `V`, `OMEGA`, `Q`, `Z3` and `PSL` were **not** <br>
rows for `T`/`U`/`V`/`OMEGA` are present in the info file, but the adjusted output was never produced.

### Months are assigned by record position, not from the timestamps

The adjusted files carry *paleo* month boundaries on their time axis. Decoded, the first year
reads

```
0401-01-27, 0401-02-28, 0401-03-31, 0401-05-01 05:36, 0401-05-29, 0401-06-26, ...
```

so April is stamped five and a half hours into May and `.dt.month` returns **no April and two
Mays**, silently folding two months into one. An earlier version of this notebook nudged the
axis back two days to repair that, which works on this file but clears the boundary by only a
few hours — a re-run at a different age or tolerance moves it, and the failure is silent.

`cal_adjust.f90` writes exactly twelve records per simulation year in paleo Jan–Dec order, so
months are assigned by position instead and the timestamps are used for nothing. See
`assign_paleocal_month_year()` in `scripts/py_functions/icesm_funcs.py`, and the assertion in
the processing cell below that checks the two approaches still agree on this file.

In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline, core_site_boxes
from icesm_funcs import (DAT_META, assign_cesm_month_year, assign_paleocal_month_year,
                         derive_dat, monthly_climatology, seasonal_means)
from stats_funcs import sigtest, sigtest2n, mask_insignificant

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
import pandas as pd 
from scipy.stats import pearsonr

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, ListedColormap, LinearSegmentedColormap
from matplotlib import cm
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here, HPC has no path to OneDrive, so this is a separate directory from the
# laptop-side FIG_OUTPUT_DIR (config/paths.env.example, proxy-side only)
# sync the two by hand (rsync/scp) when picking work back up on the laptop.
# Override the root with HPC_FIG_OUTPUT_DIR; defaults to an outputs/ subdir under the repo-root/notebooks/ dir
opath=os.path.join(os.environ.get('HPC_FIG_OUTPUT_DIR', f'{dpath0}/nam-interglacial-dD/notebooks/outputs'))
os.makedirs(opath, exist_ok=True)

In [ ]:
# --- LOAD COMPARISON DATA --- #

# IMERG precipitation
climo_filen = f'{dpath0}/obs_data/imerg.gn.2001-2018.climo.nc'
if os.path.exists(climo_filen):
    imerg = xr.open_dataset(climo_filen).precipitation
else:
    filen = f'{dpath0}/obs_data/imerg.gn.timeseries.2001-2018.nc'
    ds = xr.open_dataset(filen).precipitation.transpose('time','lat','lon').groupby("time.month").mean(dim='time') * 24 # convert from mm/hr to mm/day
    ds.attrs['units'] = 'mm/day'
    ds.attrs['Units'] = 'mm/day'
    # Convert lons to 0:360 convention
    imerg = lonFlip(ds)
    imerg.attrs['source_filename'] = 'imerg.gn.timeseries.2001-2018.nc'
    imerg.to_netcdf(climo_filen, mode='w')

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# Proxy dDp timeslice mean values.
# the model quantity these are compared against is itself dD of precipitation -- see data/processed/README.md.
proxydD = pd.read_csv('../data/processed/timeslice_mean_proxy_dDp.csv')

In [ ]:
### +++ SET FILE PATH INFO FOR iCESM1.2 OUTPUT +++ ###

# PI: the per-year (years 801-900) subset produced by scripts/nco/subset_tseries.sh, which
# writes into this repo's own data/raw/ (not $WORK_DATA_DIR) -- see data/README.md
#
# LIG: the paleo-calendar-adjusted files, years 401-500
# See the notebook header for their provenance
#
# Both the monthly climatology (`dat_climo`, below) and the per-year sample used for
# significance testing (`dat_ts`, below) come from these same files, for both cases.

iso_varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
             'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']
raw_varns = iso_varns + ['PRECC', 'PRECL', 'TS']

# The adjusted LIG output is a PaleoCalAdjust product, not a product of this repo's pipeline
lig_adj_dir = os.environ.get('LIG_ADJUSTED_DIR',
                             f'{dpath0}/nam-interglacial-dD/data/raw')
lig_stem = 'Amon_CESM1.2_LIG127k_146031-182500_cal_adj'   # 146031-182500 = days, i.e. years 401-500

files = {'pi': {}, 'lig': {}}
for varn in raw_varns:
    files['pi'][varn]  = f'{module_path}/data/raw/{varn}.iPI.01.0801-0900.tseries.nc'
    files['lig'][varn] = f'{lig_adj_dir}/{varn}_{lig_stem}.nc'

In [ ]:
### +++ PROCESS iCESM1.2 OUTPUT +++ ###

cases = ['pi', 'lig']

# First simulation year of the LIG record. PI's years come off its timestamps; the LIG's are
# assigned by record position (see below), so its first year has to be stated outright.
LIG_FIRST_YEAR = 401

#== load per-year raw variables (1200 monthly records each: PI years 801-900, LIG years 401-500)
raw = {case: {} for case in cases}
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
for case in cases:
    for varn in raw_varns:
        da = xr.open_dataset(files[case][varn], decode_times=time_coder)[varn]
        if case == 'pi':
            # CESM h0 timestamps the END of each averaging period (Jan's mean is stamped
            # 0801-02-01), so the axis is nudged back two days before .dt is trusted.
            raw[case][varn] = assign_cesm_month_year(da)
        else:
            # Months come from record position, NOT from these timestamps -- the paleo calendar
            # stamps April at 0401-05-01 05:36, so .dt.month returns no April and two Mays.
            # See the notebook header and icesm_funcs.assign_paleocal_month_year().
            raw[case][varn] = assign_paleocal_month_year(da, first_year=LIG_FIRST_YEAR)

# Guard: on this vintage of the adjusted files the positional assignment and the old two-day
# nudge agree exactly, month for month. They are not guaranteed to -- the nudge clears the April
# boundary by about five hours -- so if a re-run of cal_adjust.f90 ever moves a boundary further,
# this fails here rather than silently relabelling a season. Positional is the correct one.
_probe = raw['lig']['PRECC']
_nudged = (_probe.time - pd.Timedelta(days=2)).dt.month.values
assert (_probe['month'].values == _nudged).all(), (
    'positional month assignment disagrees with the two-day-nudge heuristic -- the adjusted '
    'file is a different vintage than the one documented in the header; inspect it before going on')
print(f"Loaded {_probe.sizes['time']} monthly records per variable per case; "
      f"LIG years {int(_probe['year'].min())}-{int(_probe['year'].max())}.")


#== construct processed data dictionaries
dat_varns = ['TS', 'PRECC', 'PRECL', 'PRECT', 'dDp', 'd18Op']

# per-year derived data, for significance testing (see the significance-testing section below)
dat_ts = derive_dat(raw, cases, dat_varns, time_dim='time')

# monthly climatology
raw_clim = monthly_climatology(raw, cases, raw_varns)
dat_climo = derive_dat(raw_clim, cases, dat_varns, time_dim='month')
print('Calculated monthly climatologies.')

# seasonal and annual averaging
seasons = ['ann', 'jas']
seas_mean, ann_seas_mean = seasonal_means(dat_ts, cases, dat_varns, seasons)
print('Calculated annual and seasonal averages.')


#== iCESM1.2 topography
# The PI/modern boundary condition, NOT a paleo topography -- unlike LGM_analyses.ipynb, which
# contours the 21 ka ice-sheet topography. The LIG's ice sheets and land surface are essentially
# modern, so the same field applies to both cases here. Written by scripts/nco/subset_tseries.sh.
PHIS = xr.open_dataset(f'{module_path}/data/raw/PHIS.iPI.01.constant.nc').PHIS.isel(time=0)
LANDFR = xr.open_dataset(f'{module_path}/data/raw/LANDFRAC.iPI.01.constant.nc').LANDFRAC.isel(time=0)
# the available variable is "surface geopotential" in units m2/s2
# approximate the surface geometric elevation by dividing by gravitational acceleration
g = 9.80665 # m/s2
zsurf = PHIS / g
zsurf.attrs = {'units': 'm', 'long_name': 'surface elevation',
               'source_file': 'PHIS.iPI.01.constant.nc'}

## Statistical significance of LIG$-$PI differences

The cell below runs `sigtest2n()` (Welch's, unpaired -- `pi` and `lig` are independent
simulations, not paired samples, and they are drawn from different centuries of their
respective runs) to get `lig_pi_diff`/`lig_pi_diff_mask`/`lig_pi_ptvals` for every field.
Both samples are 100 simulation years.

In [ ]:
### +++ LIG-PI SIGNIFICANCE TESTING +++ ###

lig_pi_diff = {varn: {} for varn in dat_varns}
lig_pi_diff_mask = {varn: {} for varn in dat_varns}
lig_pi_ptvals = {varn: {} for varn in dat_varns}

print('Significance testing LIG vs PI for:')
for varn in dat_varns:
    print(f'...{varn}')
    for season in seasons:
        diff_, diff_mask_, ptvals_ = sigtest2n(
            ann_seas_mean['lig'][varn][season], ann_seas_mean['pi'][varn][season],
            seas_mean['lig'][varn][season], seas_mean['pi'][varn][season]
        )
        lig_pi_diff[varn][season] = diff_
        lig_pi_diff_mask[varn][season] = diff_mask_
        lig_pi_ptvals[varn][season] = ptvals_

print('Done.')

## Core-site box means (LIG$-$PI)

In [ ]:
# --- CORE-SITE MEAN LIG-PI dD_precip --- #

# The boxes are defined once, in scripts/py_functions/domain_funcs.py -> core_site_boxes(), and
# are the same object drawn on the maps below (ax.add_geometries(...)) -- so the region shown
# and the region averaged here cannot drift apart. Same box definitions and same cos(lat)
# weighted-mean approach as swna_modern_climatology.ipynb and LGM_analyses.ipynb.
#
# The model grid is 0:360 in longitude; core_site_boxes() is -180:180 (matching the proxy lon/
# lat columns and how boxes are drawn on these PlateCarree maps), so bounds are converted with
# `% 360` before slicing.

proxy_varns = ['dDp']

site_boxes = core_site_boxes()
site_diff  = {site: { varn:{} for varn in proxy_varns } for site in site_boxes}

for site, poly in site_boxes.items():
    w, s, e, n = poly.bounds  # shapely: (min_lon, min_lat, max_lon, max_lat)
    for varn in proxy_varns:
        for season in seasons:
            box = lig_pi_diff[varn][season].sel(lon=slice(w % 360, e % 360), lat=slice(s, n))
            weights = np.cos(np.deg2rad(box.lat))
            site_diff[site][varn][season] = float(box.weighted(weights).mean(('lat', 'lon')))

for varn in proxy_varns:
    print(f'Model LIG-PI {varn} [per mil] -- cos(lat)-weighted mean over the averaging box around each core site')
    print(f"{'season':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
    for season in seasons:
        print(f'{season:>8}' + ''.join(f"{site_diff[site][varn][season]:>12.2f}" for site in site_boxes))


# Calculate LIG-Late Holocene dDprecip differences for the proxy records and print summary.
# late_holocene_dD is the 0-4 ka window, which is the one every published anomaly here reflects
# -- NOT holocene_dD (0-11.7 ka), which flips the sign of the DSDP-480/479 LIG anomaly.
core_dDdiff = proxydD['lig_dD'].values - proxydD['late_holocene_dD'].values

print('\nProxy LIG-LH dDp [per mil] -- at each specific core site')
print(f"{'':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
print(f"{'':>8}" + ''.join(f'{value:>12.2f}' for value in core_dDdiff))

In [ ]:
#=== PATTERN CORRELATION

# Computed here and carried into the figure annotation below in `pattern_r`
#
# Correlations use the UNMASKED differences on purpose: significance masking blanks different
# cells in each field, so correlating the masked pair would compare two different sets of grid
# points.

corr_bnds = dict(lon=slice(240, 277.5), lat=slice(10, 36))
-120., -82.5, 10., 36.
pattern_r = {}
a = lig_pi_diff['dDp'][season].sel(**corr_bnds).values.flatten()
for varn, label in [('PRECT', 'prec'), ('TS', 'Tsurf')]:
    b = lig_pi_diff[varn][season].sel(**corr_bnds).values.flatten()
    r, p = pearsonr(a, b)
    pattern_r[varn] = r
    print(f"dDp & {label:<6}| pattern correlation: r = {r:.3f}, p = {p:.3e}")

# FIGS

In [ ]:
### +++ USER-DEFINED FIG SETTINGS AND INPUTS +++ ###

# season plotted by the figures below -- select any key in `seasons` (set in the processing cell above)
season = 'jas'

# plot specs
bbox     = {'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw  = {'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1 = {'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'bottom'}
text_kw2 = {'color':'k', 'weight':'normal', 'size':12, 'ha':'right', 'va':'bottom'}
letters = np.array(['a','b','c'])

# map specs
trans = ccrs.PlateCarree()
proj  = ccrs.PlateCarree()
map_bnds = [-120., -82.5, 10., 36.]

# NAM domain outline, shared with swna_modern_climatology.ipynb and LGM_analyses.ipynb
# (scripts/py_functions/domain_funcs.py). site_boxes comes from the core-site box means cell
# above -- same object, so what's drawn here and what's averaged there can't drift apart.
nam_domain = nam_domain_outline()

# Lat/Lon vars
core_lons,  core_lats  = proxydD['lon'].values, proxydD['lat'].values
model_lons, model_lats = lig_pi_diff['PRECT'][season].lon, lig_pi_diff['PRECT'][season].lat

# construct dictionaries of per-panel model data + color specs
panels = [
    dict(title=r'$\mathbf{\Delta}$ Precipitation',                 # Δ Precipitation
         data=lig_pi_diff_mask['PRECT'][season],
         cmap=get_settings(field='precip', diff=True)[0], vmin=-2.5, vmax=2.5, nlevels=21,
         cbar_ticks=[-2, -1, 0, 1, 2], cbar_label='[mm day$^{-1}$]',
         proxy_colored=False,  # True to shade core sites on color map
         pattern_corr=pattern_r['PRECT']
        ),
    dict(title=r'$\mathbf{\Delta}\ \mathbf{\delta D_{precip}}$',   # Δ δD_precip
         data=lig_pi_diff_mask['dDp'][season],
         cmap=cm.RdBu_r, vmin=-3, vmax=3, nlevels=25,
         cbar_ticks=[-3, -2, -1, 0, 1, 2, 3], cbar_label=u'[‰]',
         proxy_colored=True,   # True to shade core site markers on cmap
         pattern_corr=None
        ),
    dict(title=r'$\mathbf{\Delta}$ Surface Temperature',           # Δ TS
         data=lig_pi_diff_mask['TS'][season],
         cmap=get_settings(field='temp', diff=True)[0], vmin=-6, vmax=6, nlevels=25,
         cbar_ticks=[-6, -4, -2, 0, 2, 4, 6], cbar_label='[K]',
         proxy_colored=False,  # True to shade core site markers on cmap
         pattern_corr=pattern_r['TS']
        ),
]
# colormap normalization
for p in panels:
    p['norm'] = mpl.colors.BoundaryNorm(np.linspace(p['vmin'], p['vmax'], p['nlevels']), p['cmap'].N)

In [ ]:
### +++ LIG-PI CLIMATOLOGY DIFFERENCES FIGURE +++ ###

fig, ax = plt.subplots(nrows=1, ncols=3,
                       figsize=(16,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5,1,' iCESM1.2 LIG (127ka) $-$ PI differences : '+season, **text_kw)

for i, (axi, p) in enumerate(zip(ax, panels)):

    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)

    # not sure why an r=0.00 is plotting on the second panel, it should have no pattern correlation value render
    if p['pattern_corr'] is not None:
        axi.text(map_bnds[1] - 1, map_bnds[3] + 0.1, f'$r={p["pattern_corr"]:.2f}$', **text_kw2)

    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'], cmap=p['cmap'], norm=p['norm'], transform=trans)

    # plot core site markers
    if p['proxy_colored']:
        for j, site_data in enumerate(core_dDdiff):
            axi.scatter(x=core_lons[j], y=core_lats[j], c=site_data,
                        cmap=p['cmap'], norm=p['norm'], alpha=1, ec='k', s=150,
                        transform=trans, zorder=100)
            axi.text(core_lons[j]-1.1, core_lats[j], f'{site_data:.1f}‰',
                     fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)
        # add averaging area boxes around core sites
        axi.add_geometries(list(site_boxes.values()), crs=trans, fc='none', ec='k', lw=1,
                            linestyle='--', zorder=9)
    else:
        axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to all three panels
    # model topography (PI/modern boundary condition -- see the processing cell)
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# colorbars
for i, p in enumerate(panels):
    cax = fig.add_axes([0.05 + i*0.325, 0, 0.275, 0.05])
    cbar = fig.colorbar(p['cf'], ticks=p['cbar_ticks'], orientation='horizontal', extend='both', cax=cax)
    cbar.set_label(p['cbar_label'], weight='normal', labelpad=5, rotation=0)
    cbar.ax.tick_params(labelsize=10)
    for tick in cbar.ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('normal')

# save output
plt.savefig(os.path.join(opath, 'LIG-PI_icesm1p2_diffs.png'), dpi=1200, bbox_inches='tight')

## Precip

### LIG-PI Convective vs. Large-Scale Precip Differences

In [ ]:
### +++ LIG-PI CONVECTIVE vs. LARGE-SCALE PRECIP DIFFERENCES FIGURE +++ ###

# --- Plot-specific settings --- #
# Map/text/domain specs are deliberately NOT redefined here -- they are reused from the fig
# settings cell so the two figures cannot drift apart.

# both panels share one cmap/norm and one colorbar
pc_cmap, _, _, _ = get_settings(field='precip', diff=True)
pc_vmin = -2.5
pc_vmax = 2.5
pc_levels = 21
pc_norm = mpl.colors.BoundaryNorm(np.linspace(pc_vmin, pc_vmax, pc_levels), pc_cmap.N)

pc_panels = [
    dict(title='PRECC  (convective)',   data=lig_pi_diff_mask['PRECC'][season]),
    dict(title='PRECL  (large-scale)',  data=lig_pi_diff_mask['PRECL'][season]),
]

# --- Make plot --- #
fig, ax = plt.subplots(nrows=1, ncols=2,
                       figsize=(11,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5, 1, f' iCESM1.2 LIG (127ka) $-$ PI precipitation partition : {season}', **text_kw)

for i, (axi, p) in enumerate(zip(ax, pc_panels)):

    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)

    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'],
                             cmap=pc_cmap, norm=pc_norm, transform=trans)

    # core site markers
    axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to both panels
    # model topography
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# one shared colorbar -- both panels use the same cmap/norm
cax = fig.add_axes([0.15, -.05, 0.7, 0.05])
cbar = fig.colorbar(pc_panels[-1]['cf'], ticks=[-2, -1, 0, 1, 2],
                    orientation='horizontal', extend='both', cax=cax)
cbar.set_label(r'$\Delta$ Precipitation [mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar.ax.tick_params(labelsize=10)

# save output
plt.savefig(os.path.join(opath, 'LIG-PI_icesm1p2_diffs_precc_precl.png'), dpi=1200, bbox_inches='tight')

### Annual Precip Cycle

In [ ]:
### +++ ANNUAL PRECIPITATION CYCLE OVER THE NAM CORE REGION +++ ###

# Averaging region, in the model's 0:360 convention. Not the nam_domain polygon: this is the
# rectangular NAM core box the annual-cycle comparison has always used, and the label below
# quotes its bounds.
lat_min, lat_max = 18, 33
lon_min, lon_max = 247, 256

region = dict(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))

nam_region = {'obs': imerg.sel(**region).mean(dim=['lat','lon'])}
for case in cases:
    nam_region[case] = dat_climo[case]['PRECT'].sel(**region).mean(dim=['lat','lon'])

# plot specs
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 'x-large'} 
title_kw = {'size': 'xx-large', 'weight': 'bold',  'color': 'k', 'ha':'center','va':'bottom'}
patch_kw = {'ec':'beige', 'lw':1, 'ls':'-', 'fc':'beige', 'alpha':0.5}
legend_prop={'size':'x-large', 'weight':'bold'}
legend_kw={'labelcolor':'linecolor', 'ncols':1, 'frameon':False}
line_cols = {'obs':'k', 'pi':'peru', 'lig':'#9a0200'}
labels    = {'obs':'IMERG', 'pi':'PI', 'lig':'LIG'}
idx = np.arange(12)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9,6), layout='constrained')
ax.text(5.5, 6.15, f'MONTHLY MEAN RAINFALL [{lat_min}°N$-${lat_max}°N, '
                   f'{360-lon_max}°W$-${360-lon_min}°W]', rotation=0, **title_kw)

for key in ['obs'] + cases:
    ax.plot(idx, nam_region[key], c=line_cols[key], ls='-', lw=2, label=labels[key])

# shade the season the difference figures above average over
season_months = get_season(season=season)   # 0-based positional indices
pp = plt.Rectangle((season_months[0], 0), len(season_months)-1, 7, zorder=1, label='_Hidden', **patch_kw)
ax.add_patch(pp)

ax.set(xlim=[0, 11], ylim=[0,6.1])
ax.set_xticks(idx)
ax.set_xticklabels(['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC'])
ax.set_yticks([1,2,3,4,5,6])
ax.set_ylabel('[mm/day]', weight='normal', size='x-large')
ax.tick_params(**tkw)

ax.legend(loc=2, prop=legend_prop, **legend_kw)

### Monthly Precip Climatology

In [ ]:
### +++ MONTHLY PRECIPITATION CLIMATOLOGY, ONE CASE +++ ###

# Exploratory, not a manuscript figure -- swna_modern_climatology.ipynb covers the modern
# seasonal cycle in publication form. Kept here to eyeball the simulated monsoon's onset and
# retreat; switch `climo_case` to 'lig' to see the same for the interglacial.
climo_case = 'pi'

# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}

# colormap specs
cmap2=plt.colormaps['Blues']
vmin2=0
vmax2=12
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)

# map/text/domain specs are reused from the fig settings cell above
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,f'iCESM1.2 {climo_case.upper()} Precip Climo', **font_kw)

for i, axi in enumerate(axes.flat):
    axi.text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i], **font_kw)
    cf = axi.pcolormesh(model_lons, model_lats, dat_climo[climo_case]['PRECT'].isel(month=i),
                        cmap=cmap2, norm=norm2, transform=trans)
    axi.coastlines(lw=1)
    axi.add_feature(cfeature.BORDERS)
    # NAM domain polygon outline -- the same object the difference figures draw
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    axi.set_extent(map_bnds, crs=trans)
    axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)

cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.025])
cbar = fig.colorbar(cf, orientation='horizontal', extend='max', cax=cbar_ax)
cbar.set_label('[mm day$^{-1}$]', labelpad=5, size=14, rotation=0)
cbar.ax.tick_params(labelsize=14)

## Isotopes

In [ ]:
### +++ dD_precip CLIMATOLOGIES, PI vs LIG +++ ###

# The absolute fields behind the difference panel above. Map/text/domain specs are reused from
# the fig settings cell.

# isotopes cmap
icmap = cm.RdYlBu_r
ivmin, ivmax = -12, 0
inorm = mpl.colors.BoundaryNorm(np.linspace(ivmin, ivmax, 25), icmap.N)

titles = {'pi': 'PI', 'lig': 'LIG'}

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5, 1, f'iCESM1.2 PI & LIG (127ka) $\\delta D_{{precip}}$ : {season}', **text_kw)

for i, (axi, case) in enumerate(zip(ax, cases)):
    cf = axi.pcolormesh(model_lons, model_lats, seas_mean[case]['dDp'][season],
                        cmap=icmap, norm=inorm, transform=trans)

    # add core location scatter points
    axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, titles[case], **text_kw)
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# one colorbar -- both panels share a cmap/norm. (There used to be two, both built from the
# same mappable, which drew the identical scale twice.)
cax = fig.add_axes([0.15, -.05, 0.7, 0.05])
cbar = fig.colorbar(cf, ticks=np.arange(ivmin, ivmax+1, 2), orientation='horizontal', extend='min', cax=cax)
cbar.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar.ax.tick_params(labelsize=10)

## Temperature

In [ ]:
### +++ SURFACE TEMPERATURE: PI, LIG, AND THE DIFFERENCE +++ ###

# Map/text/domain specs are reused from the fig settings cell.

# climatology cmap
cmap,_,_,_ = get_settings(field='temp', diff=False)
norm = mpl.colors.BoundaryNorm(np.linspace(15, 35, 21), cmap.N)
# difference cmap -- the same one panel (c) of the difference figure uses
dcmap, dnorm = panels[2]['cmap'], panels[2]['norm']

ts_panels = [
    dict(title='PI',        data=seas_mean['pi']['TS'][season]-273.15,  cmap=cmap,  norm=norm),
    dict(title='LIG',       data=seas_mean['lig']['TS'][season]-273.15, cmap=cmap,  norm=norm),
    dict(title='LIG$-$PI',  data=lig_pi_diff_mask['TS'][season],        cmap=dcmap, norm=dnorm),
]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16,4.5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5, 1, f'iCESM1.2 surface temperature : {season}', **text_kw)

for i, (axi, p) in enumerate(zip(ax, ts_panels)):
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'], cmap=p['cmap'], norm=p['norm'], transform=trans)
    axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# two colorbars: one shared by the PI/LIG climatology panels, one for the difference
cax1 = fig.add_axes([0.05, 0, 0.6, 0.05])
cbar1 = fig.colorbar(ts_panels[0]['cf'], ticks=np.arange(15, 40, 5), orientation='horizontal', extend='both', cax=cax1)
cbar1.set_label(u'[°C]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)

cax2 = fig.add_axes([0.7, 0, 0.275, 0.05])
cbar2 = fig.colorbar(ts_panels[2]['cf'], ticks=panels[2]['cbar_ticks'], orientation='horizontal', extend='both', cax=cax2)
cbar2.set_label('$\\Delta$ [K]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)